Bruker pip for å laste ned nødvendige biblioteker.

<!-- Dersom du er på windows eller bruker anaconda, er det mulig at du må installere noen av de manuelt igjennom anaconda programmet. -->

In [ ]:
%pip install lonboard geopandas matplotlib pyarrow folium mapclassify scikit-learn osmnx[all]

## Vektor Analayse med Tilfluktsrom og kommuner

Dataset er lastet ned fra GeoNorge

| Dataset | URL |
| --- | --- |
| Tilfluktsrom | https://kartkatalog.geonorge.no/metadata/tilfluktsrom-offentlige/dbae9aae-10e7-4b75-8d67-7f0e8828f3d8 |
| Kommuner | https://kartkatalog.geonorge.no/metadata/administrative-enheter-kommuner/041f1e6e-bdbc-4091-b48f-8a5990f3cc5b |

Importerer nødvendige biblioteker for å arbeide med geodata og visualisering.

Deretter laster vi inn datasettene for tilfluktsrom og kommuner, og konverterer koordinatsystemet til UTM sone 33N (EPSG:25833) for å kunne arbeide med dem i GeoPandas.

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
from pathlib import Path
from lonboard import viz

plt.rcParams['figure.dpi'] = 150
 
DATASET_PATH = Path('../dataset')
BUNKER_DATASET_PATH = DATASET_PATH / 'tilfluktsrom.parquet'
KOMMUNE_DATASET_PATH = DATASET_PATH / 'kommuner.parquet'

bunker_gdf = gpd.read_parquet(BUNKER_DATASET_PATH)
kommune_gdf = gpd.read_parquet(KOMMUNE_DATASET_PATH)

bunker_gdf = bunker_gdf.to_crs("EPSG:25833")
kommune_gdf = kommune_gdf.to_crs("EPSG:25833")

<p>GeoPandas og Python har problemer med å arbeide med tidsstempel objektet i datasettene, så vi må konvertere disse til en streng først.

<cite>https://stackoverflow.com/questions/49243736/how-do-i-handle-object-of-type-timestamp-is-not-json-serializable-in-python</cite>
</p>

In [ ]:
for name in ["oppdateringsdato", "datafangstdato", "datauttaksdato"]:
    if name in kommune_gdf.columns:
        kommune_gdf[name] = kommune_gdf[name].astype(str)


Lager et interaktivt kart med Folium som viser alle kommunegrenser og tilfluktsrom. Kartet gir en visuell oversikt over den geografiske fordelingen av tilfluktsrom på tvers av kommunene.

In [ ]:
m = folium.Map(location=[58.159026, 8.017441], zoom_start=12)

kommune_gdf_4326 = kommune_gdf.to_crs("EPSG:4326")
bunker_gdf_4326 = bunker_gdf.to_crs("EPSG:4326")

viz(kommune_gdf_4326)

folium.GeoJson(
    kommune_gdf,
    name="Kommuner",
    style_function=lambda _: {"color": "blue", "weight": 1, "fillOpacity": 0.05},
    tooltip=folium.GeoJsonTooltip(fields=["kommunenavn"], aliases=["Kommune:"])
).add_to(m)

bunker_layer = folium.FeatureGroup(name="Tilfluktsrom")
for _, r in bunker_gdf_4326.iterrows():
    folium.CircleMarker(
        location=[r.geom.y, r.geom.x],
        radius=3,
        color="red",
        fill=True,
        fill_opacity=0.8,
        popup=f"{r.get('adresse', 'Ukjent adresse')}<br>Plasser: {r.get('plasser', 'N/A')}"
    ).add_to(bunker_layer)

bunker_layer.add_to(m)
folium.LayerControl().add_to(m)

m

Filtrerer datasettet til én bestemt kommune (Oslo) for å fokusere analysen. Plottet bekrefter at vi har valgt riktig geografisk område.

In [ ]:
valgt_kommune = "Oslo"
kommune_filtrert = kommune_gdf[kommune_gdf['kommunenavn'] == valgt_kommune]
kommune_filtrert.plot()

Oppretter en 1 km buffer rundt hvert tilfluktsrom. Buffersonen representerer det arealet som anses å ha tilgang til et tilfluktsrom, og brukes videre til å identifisere hvilke områder som ikke har dekning.

In [ ]:
def format_distance(metres: int) -> str:
    if metres >= 1000:
        return f"{metres / 1000:.1f} km"
    else:
        return f"{metres} m"

BUFFER_SIZE = 1000
BUFFER_SIZE_STR = format_distance(BUFFER_SIZE)

bunker_buffer_gdf = gpd.GeoDataFrame(geometry=bunker_gdf.geometry.buffer(BUFFER_SIZE))

Visualiserer buffersonene på et interaktivt kart. De oransje sonene viser hvilke områder som ligger innenfor 1 km fra et tilfluktsrom, og gir en umiddelbar forståelse av dekningsgraden.

In [ ]:
m = folium.Map(location=[58.159026, 8.017441], zoom_start=12)

viz(kommune_gdf_4326)

folium.GeoJson(
    kommune_gdf_4326,
    name="Kommuner",
    style_function=lambda _: {"color": "blue", "weight": 1, "fillOpacity": 0.05},
    tooltip=folium.GeoJsonTooltip(fields=["kommunenavn"], aliases=["Kommune:"])
).add_to(m)

folium.GeoJson(
    bunker_buffer_gdf.to_crs("EPSG:4326"),
    name="1 km buffer",
    style_function=lambda _: {"color": "orange", "weight": 1, "fillOpacity": 0.06},
    tooltip=folium.GeoJsonTooltip(fields=[], aliases=[])
).add_to(m)

bunker_layer = folium.FeatureGroup(name="Tilfluktsrom")
for _, r in bunker_gdf_4326.iterrows():
    folium.CircleMarker(
        location=[r.geom.y, r.geom.x],
        radius=3,
        color="red",
        fill=True,
        fill_opacity=0.8,
        popup=f"{r.get('adresse', 'Ukjent adresse')}<br>Plasser: {r.get('plasser', 'N/A')}"
    ).add_to(bunker_layer)

bunker_layer.add_to(m)
folium.LayerControl().add_to(m)

m

Utfører en romlig overlay for å finne arealer i kommunen som **ikke** dekkes av noen buffer. Kartet viser de fargede områdene som mangler nærhet til et tilfluktsrom, og statistikken angir nøyaktig andel av kommunens areal uten dekning.

In [ ]:
from matplotlib.legend import Legend
from matplotlib.collections import PatchCollection
from matplotlib.legend_handler import HandlerPolyCollection

# https://github.com/geopandas/geopandas/issues/660#issuecomment-3312202400
Legend.update_default_handler_map({PatchCollection: HandlerPolyCollection()})

uten_dekning = kommune_filtrert.overlay(bunker_buffer_gdf, how="difference")

# Areal i km², CRS er meterbasert på grunn av EPSG:25833
kommune_areal_km2 = kommune_filtrert.geom.area.sum() / 1_000_000
uten_dekning_areal_km2 = uten_dekning.geom.area.sum() / 1_000_000
dekning_pct = 100 * (1 - uten_dekning_areal_km2 / kommune_areal_km2)

fig, ax = plt.subplots(figsize=(9, 9))

# Kommune grense i bakgrunnen
kommune_filtrert.boundary.plot(ax=ax, color="black", linewidth=1.2, label=f"{valgt_kommune} grense")

# Områder uten dekning
uten_dekning.plot(
    ax=ax,
    color="#f97316",
    edgecolor="#7c2d12",
    linewidth=0.8,
    alpha=0.75,
    label=f"Uten dekning ({BUFFER_SIZE_STR} fra tilfluktsrom)"
)

ax.set_title(
    f"Tilfluktsromdekning i {valgt_kommune}\n"
    f"Fargede områder er uten dekning og mer enn {BUFFER_SIZE_STR} fra nærmeste tilfluktsrom\n"
    f"Uten dekning: {uten_dekning_areal_km2:.1f} km² | Dekning: {dekning_pct:.1f}%",
    fontsize=13
)
ax.set_axis_off()
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

Kobler hvert tilfluktsrom til kommunen det befinner seg i ved hjelp av en romlig join `sjoin`. Resultatet gjør det mulig å telle og analysere tilfluktsrom per kommune i neste steg.

In [ ]:
joined = gpd.sjoin(bunker_gdf, kommune_gdf, how="left", predicate="within")
joined.head()

Grupperer dataene per kommune og teller antall tilfluktsrom i hver. Tabellen gir en oversikt over fordelingen av tilfluktsrom på tvers av alle kommuner i datasettet.

In [ ]:
bunkere_per_kommune = joined.groupby("kommunenavn").size().reset_index(name="antall_bunkere")

with pd.option_context('display.max_rows', None):
    print(bunkere_per_kommune)

## Raster Analyse

Dataset er lastet ned fra GeoNorge og inneholder høydedata for Kristiansand-området. Link til dataset er https://kartkatalog.geonorge.no/metadata/hoeydedatano/91bd03b1-54b5-4393-9fd0-3c927b4bb608 og datasettet heter `DTM 10 Terrengmodell (UTM33)` med geografisk område 6400-1.

Datasettet er inkludert i GitHub-repoet for denne oppgaven som en ZIP fil, den er ikke pakket ut i repoet fordi filstørrelsen er for stor. For å bruke datasettet må du pakke ut ZIP filen `raster_hoyedata_6400-1_Celle_25833_DTM10UTM33_DEM.zip` og plassere filen i dataset mappen.

### Konvertere DEM til TIF

<p>Bruke GDAL for å konvertere DEM-filen til et mer håndterlig format (TIF) for videre analyse.
<cite>https://gdal.org/en/stable/programs/gdal_translate.html</cite>
</p>

```bash
gdal_translate -of GTiff dataset/6400_1_10m_z33.dem dataset/raster.tif
```
<img src="bilder/raster.png" alt="Raster bilde" width="481" />

### Beregne slope

<p>Bruke GDALs `gdaldem`-verktøy for å beregne slope (helling) fra det konverterte rasteret.
<cite>https://gdal.org/en/stable/programs/gdaldem.html</cite>
</p>

```bash
gdaldem slope dataset/raster.tif dataset/slope.tif
```

<img src="bilder/slope.png" alt="Slope bilde" width="481" />

<p>Deretter beregne bratte områder som har en slope større enn 30 grader.
<cite>https://gdal.org/en/stable/programs/gdal_calc.html</cite>
</p>

```bash
gdal_calc.py -A dataset/slope.tif --outfile=dataset/slope_30.tif --calc="A>30"
```

<img src="bilder/slope_30.png" alt="Slope > 30 bilde" width="481" />

### Polygoniser slope

<p>Bruke GDALs `gdal_polygonize`-verktøy for å konvertere de bratte områdene (slope > 30) til polygoner. Dette gjør det mulig å visualisere og analysere de bratte områdene som vektorobjekter.
<cite>https://gdal.org/en/stable/programs/gdal_polygonize.html</cite>
</p>

```bash
gdal_polygonize.py dataset/slope_30.tif -f "GeoJSON" dataset/slope_30.geojson
```

<img src="bilder/slope_30_polygon.png" alt="Slope > 30 polygoner" width="481" />

### Hillshade

<p>Bruke GDALs `gdaldem`-verktøy igjen for å beregne hillshade, som gir en skyggeeffekt basert på terrengets orientering og solens posisjon.
<cite>https://gdal.org/en/stable/programs/gdaldem.html</cite>
</p>

```bash
gdaldem hillshade -az 315 -alt 45 dataset/raster.tif dataset/hillshade1.tif
gdaldem hillshade -az 45 -alt 45 dataset/raster.tif dataset/hillshade2.tif
gdaldem hillshade -multidirectional dataset/raster.tif dataset/hillshade_multi.tif
```

#### Hillshade med sol fra nordvest

<img src="bilder/hillshade_nw.png" alt="Hillshade nordvest" width="481" />

#### Hillshade med sol fra nordøst

<img src="bilder/hillshade_ne.png" alt="Hillshade nordøst" width="481" />

In [ ]:
slope = gpd.read_file(DATASET_PATH / 'slope_30.geojson')
fig, ax = plt.subplots(figsize=(10, 10))

slope.plot(ax=ax, alpha=1.0, edgecolor="black", legend=True)

kristiansand_kommune = kommune_gdf[kommune_gdf['kommunenavn'] == 'Kristiansand']
kristiansand_kommune.boundary.plot(ax=ax, color='red', linewidth=2, label='Kristiansand kommune')

ax.set_title('Høydeskråning i Kristiansand-området', fontsize=14)
ax.set_axis_off()
ax.legend()

plt.show()